In [545]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split

from nltk.tag import pos_tag
import pandas as pd

from string import punctuation
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [546]:
#Setting Up Variable
stop_words = stopwords.words("english")
print(stop_words)

stemmer = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()



['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [547]:
#EDA
df = pd.read_csv("Reddit_Data.csv")
df.head(5)

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [548]:
df["category"].value_counts()
#Balance The Data:
#8.2 K In All Category

category
 1    15830
 0    13142
-1     8277
Name: count, dtype: int64

In [549]:
df["category"] = df["category"].replace(0, 1)

In [550]:
df["category"].value_counts()   

category
 1    28972
-1     8277
Name: count, dtype: int64

In [551]:
n_minority = df[df.category == -1].shape[0]
df_majority = df[df.category == 1].sample(n=n_minority, random_state=42, replace=False)
df_minority = df[df.category == -1]
df_balanced = (
    pd.concat([df_majority, df_minority])
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)
print(df_balanced.category.value_counts()) 

category
 1    8277
-1    8277
Name: count, dtype: int64


In [552]:
df_balanced.isna().sum()

clean_comment    27
category          0
dtype: int64

In [553]:
#Data Cleaning
df_balanced.dropna(inplace=True)

In [554]:
df_balanced.head(5)

,clean_comment,category
0,people reverse engineer equipment all the time,1
1,guys make sure you are running the main bow ar...,1
2,and when told pals back january 2014 that thes...,1
3,you think this level backtracking funny wait ...,1
4,niente volevo condividere con voi questa porn...,1


In [555]:
def get_tag(tag):
    tag = tag.lower()
    if tag.startswith('j'):
        return wordnet.ADJ
    elif tag.startswith('v'):
        return wordnet.VERB
    elif tag.startswith('n'):
        return wordnet.NOUN
    elif tag.startswith('r'):
        return wordnet.ADV
    else:
        return wordnet.NOUN


In [ ]:
def pre_process(sentence : str):
    token_list = []
    sentence = sentence.lower()
    tokens = word_tokenize(sentence)
    for token in tokens:
        if token not in stop_words and token not in punctuation:
            token_list.append(stemmer.stem(token))
    
    lemmatized_token_list = []
    
    pos_tagged_token = pos_tag(token_list)
    for token in pos_tagged_token:
        lemmatized_token = lemmatizer.lemmatize(token[0], get_tag(token[1]))
        lemmatized_token_list.append(lemmatized_token)
    return lemmatized_token_list

In [557]:
pre_processed_df = []
for sentence in df_balanced["clean_comment"]:
    pre_processed_df.append(pre_process(sentence))

In [558]:
#Split The Dataset Before Feature Extraction
X_tokens = pre_processed_df
y = df_balanced["category"].tolist()

X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
    X_tokens,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
)

In [559]:
#Load All Dataset to Pre-Process
pre_processed_df = []
for sentence in df_balanced["clean_comment"]:
    pre_processed_token = pre_process(sentence)
    pre_processed_df.append(pre_processed_token)

In [560]:
train_docs = [" ".join(tokens) for tokens in X_train_tokens]
test_docs  = [" ".join(tokens) for tokens in X_test_tokens]

In [561]:
tfidf = TfidfVectorizer(
    ngram_range=(1,1),
    max_df=0.6,
    min_df=20
)

In [562]:
X_train_tfidf = tfidf.fit_transform(train_docs)
X_test_tfidf  = tfidf.transform(test_docs)

In [563]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [564]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
y_pred_rf = rf.predict(X_test_tfidf)

In [565]:
print("=== Random Forest Results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

=== Random Forest Results ===
Accuracy: 0.8263762855414398
              precision    recall  f1-score   support

          -1       0.82      0.84      0.83      1656
           1       0.84      0.81      0.82      1650

    accuracy                           0.83      3306
   macro avg       0.83      0.83      0.83      3306
weighted avg       0.83      0.83      0.83      3306

Confusion Matrix:
 [[1398  258]
 [ 316 1334]]


In [566]:
svm = make_pipeline(
    LinearSVC(C=1.0, max_iter=5000, random_state=42, class_weight="balanced")
)
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)

d:\Anaconda\envs\nlp\lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


In [568]:
from sklearn.metrics import classification_report, accuracy_score
print("RF acc:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print("SVM acc:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

RF acc: 0.8263762855414398
              precision    recall  f1-score   support

          -1       0.82      0.84      0.83      1656
           1       0.84      0.81      0.82      1650

    accuracy                           0.83      3306
   macro avg       0.83      0.83      0.83      3306
weighted avg       0.83      0.83      0.83      3306

SVM acc: 0.8324258923169994
              precision    recall  f1-score   support

          -1       0.86      0.79      0.83      1656
           1       0.81      0.87      0.84      1650

    accuracy                           0.83      3306
   macro avg       0.83      0.83      0.83      3306
weighted avg       0.83      0.83      0.83      3306



In [ ]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

df_cleaned = df_balanced.copy()
df_cleaned.dropna(subset=["clean_comment","category"], inplace=True)

texts = df_cleaned["clean_comment"].tolist()
labels = df_cleaned["category"].tolist()

train_texts, test_texts, y_train, y_test = train_test_split(
    texts, labels,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=labels
)

def preprocess_and_join(texts):
    return [" ".join(pre_process(t)) for t in texts]

svm_pipe = Pipeline([
    ("prep", FunctionTransformer(preprocess_and_join, validate=False)),
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,1),
        max_df=0.6,
        min_df=20
    )),
    ("svc", LinearSVC(
        C=1.0,
        max_iter=5000,
        class_weight="balanced",
        random_state=42,
        dual=False
    ))
])

svm_pipe.fit(train_texts, y_train)

y_pred = svm_pipe.predict(test_texts)
from sklearn.metrics import accuracy_score, classification_report
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

with open('svm_full_pipeline.pkl', 'wb') as f:
    pickle.dump(svm_pipe, f)
print("Pipeline saved to /mnt/data/svm_full_pipeline.pkl")


Test Accuracy: 0.8324258923169994
              precision    recall  f1-score   support

          -1       0.86      0.79      0.83      1656
           1       0.81      0.87      0.84      1650

    accuracy                           0.83      3306
   macro avg       0.83      0.83      0.83      3306
weighted avg       0.83      0.83      0.83      3306

Pipeline saved to /mnt/data/svm_full_pipeline.pkl


In [ ]:
import pickle

with open('svm_full_pipeline.pkl','rb') as f:
    loaded_pipe = pickle.load(f)

new_text = "i hate you"
pred = loaded_pipe.predict([new_text])
print("Predicted label:", pred[0])

Predicted label: -1
